# Sim Lab Explorer

Analyze Sim Lab batch runs — see which Ricky ranks and SE tags are producing
the most edge against projections.

**Inputs**
- `ricky_ranked_lineups*.csv` — per-lineup detail with rank, tag, proj, actual
- `*export.csv` — SE Core / SE Spicy / SE Alt headline lineups

**Outputs** — Rank-bucket summary, tag-level summary, bar charts, scatter plot

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
# Point DATA_DIR at the folder containing your Sim Lab CSV exports.
# Restart & Run All safe.

from pathlib import Path

DATA_DIR = Path("../data/sim_lab")   # <-- change to your exports folder

## Imports

In [ ]:
import warnings

import numpy as np
import pandas as pd
from IPython.display import HTML, display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.2f}".format)

# Dark-mode table styling (matches YakOS / Smash-Bust Explorer palette)
_TABLE_CSS = """
<style>
.df-dark { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 12px; }
.df-dark table { border-collapse: collapse; width: 100%; background: #161921; color: #c8ccd4; }
.df-dark th { background: #1a1d28; color: #6b7280; text-transform: uppercase; letter-spacing: .4px;
              font-size: 10px; padding: 6px 10px; border-bottom: 1px solid #2a2d38; text-align: right; }
.df-dark td { padding: 5px 10px; border-bottom: 1px solid #1e2130; text-align: right; font-family: monospace; font-size: 11px; }
.df-dark tr:hover td { background: #1e2130; }
.df-dark td:first-child, .df-dark th:first-child { text-align: left; }
.pos { color: #22c55e; } .neg { color: #ef4444; }
</style>
"""
display(HTML(_TABLE_CSS))


def dark_table(df, title=""):
    """Render a DataFrame as a dark-mode HTML table."""
    html = '<div class="df-dark">'
    if title:
        html += f'<div style="color:#e2e5ec;font-size:14px;font-weight:600;margin-bottom:6px">{title}</div>'
    html += df.to_html(classes="", border=0, escape=False)
    html += '</div>'
    return HTML(html)

## Load Ranked Lineups

In [ ]:
lineup_files = sorted(DATA_DIR.glob("ricky_ranked_lineups*.csv"))
print(f"Found {len(lineup_files)} ranked-lineup file(s)")
for f in lineup_files:
    print(f"  {f.name}")

if lineup_files:
    lineups = pd.concat([pd.read_csv(f) for f in lineup_files], ignore_index=True)
else:
    lineups = pd.DataFrame()
    print("⚠ No ricky_ranked_lineups*.csv found in DATA_DIR")

if not lineups.empty:
    lineups["date"] = pd.to_datetime(lineups["date"], errors="coerce")
    if "diff" not in lineups.columns and {"total_actual", "total_proj"}.issubset(lineups.columns):
        lineups["diff"] = lineups["total_actual"] - lineups["total_proj"]
    print(f"\n{len(lineups):,} lineups · {lineups['date'].nunique()} dates · "
          f"ranks {lineups['ricky_rank'].min()}–{lineups['ricky_rank'].max()}")
    display(lineups.head(3))

## Load SE Exports

In [ ]:
export_files = sorted(DATA_DIR.glob("*export.csv"))
print(f"Found {len(export_files)} export file(s)")
for f in export_files:
    print(f"  {f.name}")

if export_files:
    exports = pd.concat([pd.read_csv(f) for f in export_files], ignore_index=True)
else:
    exports = pd.DataFrame()
    print("⚠ No *export.csv found in DATA_DIR")

if not exports.empty:
    # Normalize columns — UI exports use title-case
    _cmap = {
        "Date": "date", "Tag": "tag", "Rank": "rank", "Score": "score",
        "GPP Score": "gpp_score", "Ceiling": "ceiling",
        "Proj": "proj", "Actual": "actual",
        "Avg Own%": "avg_own_pct", "Salary": "salary", "Diff": "diff",
    }
    exports.rename(columns={k: v for k, v in _cmap.items() if k in exports.columns}, inplace=True)

    # Handle lowercase columns from ranked-lineups format
    _lower = {"ricky_tag": "tag", "total_proj": "proj", "total_actual": "actual",
              "total_gpp_score": "gpp_score", "total_ceil": "ceiling"}
    exports.rename(columns={k: v for k, v in _lower.items() if k in exports.columns and v not in exports.columns}, inplace=True)

    exports["date"] = pd.to_datetime(exports["date"], errors="coerce")
    if "diff" not in exports.columns and {"actual", "proj"}.issubset(exports.columns):
        exports["diff"] = exports["actual"] - exports["proj"]
    if "tag" in exports.columns:
        exports = exports[exports["tag"].str.strip().ne("") & exports["tag"].notna()].copy()

    print(f"\n{len(exports):,} tagged lineups · {exports['date'].nunique()} dates")
    display(exports.head(3))

## Rank Bucket Summary

In [ ]:
if not lineups.empty and "ricky_rank" in lineups.columns:

    def _bucket(r):
        if r <= 5:  return "1–5"
        if r <= 10: return "6–10"
        return "11–20"

    lineups["rank_bucket"] = lineups["ricky_rank"].apply(_bucket)

    _agg = {}
    for c in ["diff", "total_proj", "total_actual", "total_gpp_score"]:
        if c in lineups.columns:
            _agg[c] = ["count", "mean", "median"] if c == "diff" else ["mean"]

    rank_summary = lineups.groupby("rank_bucket", sort=False).agg(_agg)
    rank_summary.columns = [
        f"{stat}_{col}" if stat != "count" else "n_lineups"
        for col, stat in rank_summary.columns
    ]
    rank_summary = rank_summary.reindex(["1–5", "6–10", "11–20"]).dropna(how="all")
    rank_summary = rank_summary.rename(columns={
        "n_lineups": "Lineups", "mean_diff": "Avg Diff", "median_diff": "Med Diff",
        "mean_total_proj": "Avg Proj", "mean_total_actual": "Avg Actual",
        "mean_total_gpp_score": "Avg GPP",
    })
    rank_summary.index.name = "Rank"

    # Color the Diff columns
    def _color_diff(v):
        if pd.isna(v): return ""
        cls = "pos" if v >= 0 else "neg"
        return f'<span class="{cls}">{v:+.1f}</span>'

    _rs = rank_summary.copy()
    for c in ["Avg Diff", "Med Diff"]:
        if c in _rs.columns:
            _rs[c] = _rs[c].apply(_color_diff)
    for c in ["Lineups"]:
        if c in _rs.columns:
            _rs[c] = _rs[c].astype(int)
    for c in ["Avg Proj", "Avg Actual", "Avg GPP"]:
        if c in _rs.columns:
            _rs[c] = _rs[c].round(1)

    display(dark_table(_rs, "Lineup Performance by Ricky Rank Bucket"))
else:
    print("No lineup data to summarize.")

## Tag Summary by Date

In [ ]:
if not exports.empty and "tag" in exports.columns:
    _agg = {}
    for c, ops in [("diff", ["count", "mean", "median"]), ("proj", ["mean"]),
                    ("actual", ["mean"]), ("avg_own_pct", ["mean"])]:
        if c in exports.columns:
            _agg[c] = ops

    if _agg:
        tag_summary = (
            exports
            .groupby([exports["date"].dt.strftime("%Y-%m-%d"), "tag"], sort=False)
            .agg(_agg)
        )
        tag_summary.columns = [
            f"{stat}_{col}" if stat != "count" else "n_lineups"
            for col, stat in tag_summary.columns
        ]
        tag_summary = tag_summary.rename(columns={
            "n_lineups": "Lineups", "mean_diff": "Avg Diff", "median_diff": "Med Diff",
            "mean_proj": "Avg Proj", "mean_actual": "Avg Actual",
            "mean_avg_own_pct": "Avg Own%",
        })
        tag_summary.index.names = ["Date", "Tag"]

        _ts = tag_summary.copy()
        for c in ["Avg Diff", "Med Diff"]:
            if c in _ts.columns:
                _ts[c] = _ts[c].apply(lambda v: f'<span class="{"pos" if v>=0 else "neg"}">{v:+.1f}</span>' if pd.notna(v) else "")
        for c in ["Lineups"]:
            if c in _ts.columns:
                _ts[c] = _ts[c].astype(int)
        for c in ["Avg Proj", "Avg Actual", "Avg Own%"]:
            if c in _ts.columns:
                _ts[c] = _ts[c].round(1)

        display(dark_table(_ts, "SE Tag Performance by Date"))
    else:
        print("Export columns missing for aggregation.")
else:
    print("No export data to summarize.")

## Rank Bucket — Avg Diff

In [ ]:
if not lineups.empty and "rank_bucket" in lineups.columns:
    _bo = ["1–5", "6–10", "11–20"]
    _bs = lineups.groupby("rank_bucket")["diff"].agg(["mean", "median", "count"]).reindex(
        [b for b in _bo if b in lineups["rank_bucket"].unique()]
    )
    _labels = list(_bs.index)
    _means  = [round(v, 2) for v in _bs["mean"]]
    _meds   = [round(v, 2) for v in _bs["median"]]
    _counts = [int(v) for v in _bs["count"]]

    _chart = f"""
    <div style="background:#0f1117;border-radius:8px;padding:16px;max-width:700px">
      <div style="color:#e2e5ec;font-size:14px;font-weight:600;margin-bottom:10px">
        Lineup Performance by Ricky Rank Bucket</div>
      <canvas id="rankChart" height="280"></canvas>
    </div>
    <script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.7/dist/chart.umd.min.js"></script>
    <script>
    new Chart(document.getElementById('rankChart'), {{
      type: 'bar',
      data: {{
        labels: {_labels},
        datasets: [
          {{ label: 'Avg Diff', data: {_means}, backgroundColor: '#4a9eff99', borderColor: '#4a9eff', borderWidth: 1 }},
          {{ label: 'Median Diff', data: {_meds}, backgroundColor: '#ef444499', borderColor: '#ef4444', borderWidth: 1 }},
        ]
      }},
      options: {{
        responsive: true,
        plugins: {{
          legend: {{ labels: {{ color: '#6b7280', font: {{ size: 11 }} }} }},
          tooltip: {{ callbacks: {{ afterLabel: (ctx) => 'n=' + {_counts}[ctx.dataIndex] }} }}
        }},
        scales: {{
          x: {{ ticks: {{ color: '#9ca3af' }}, grid: {{ color: '#1e2130' }} }},
          y: {{ ticks: {{ color: '#9ca3af' }}, grid: {{ color: '#1e2130' }},
               title: {{ display: true, text: 'Diff (Actual − Proj)', color: '#6b7280' }} }}
        }}
      }}
    }});
    </script>
    """
    display(HTML(_chart))
else:
    print("No data for rank chart.")

## SE Tag — Avg Diff

In [ ]:
if not exports.empty and "tag" in exports.columns and "diff" in exports.columns:
    _to = ["SE Core", "SE Spicy", "SE Alt"]
    _ts = exports.groupby("tag")["diff"].agg(["mean", "median", "count"]).reindex(
        [t for t in _to if t in exports["tag"].unique()]
    )
    _labels = list(_ts.index)
    _means  = [round(v, 2) for v in _ts["mean"]]
    _meds   = [round(v, 2) for v in _ts["median"]]
    _counts = [int(v) for v in _ts["count"]]

    _chart = f"""
    <div style="background:#0f1117;border-radius:8px;padding:16px;max-width:700px">
      <div style="color:#e2e5ec;font-size:14px;font-weight:600;margin-bottom:10px">
        SE Tag Performance — Core vs Spicy vs Alt</div>
      <canvas id="tagChart" height="280"></canvas>
    </div>
    <script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.7/dist/chart.umd.min.js"></script>
    <script>
    new Chart(document.getElementById('tagChart'), {{
      type: 'bar',
      data: {{
        labels: {_labels},
        datasets: [
          {{ label: 'Avg Diff', data: {_means}, backgroundColor: '#22c55e99', borderColor: '#22c55e', borderWidth: 1 }},
          {{ label: 'Median Diff', data: {_meds}, backgroundColor: '#f59e0b99', borderColor: '#f59e0b', borderWidth: 1 }},
        ]
      }},
      options: {{
        responsive: true,
        plugins: {{
          legend: {{ labels: {{ color: '#6b7280', font: {{ size: 11 }} }} }},
          tooltip: {{ callbacks: {{ afterLabel: (ctx) => 'n=' + {_counts}[ctx.dataIndex] }} }}
        }},
        scales: {{
          x: {{ ticks: {{ color: '#9ca3af' }}, grid: {{ color: '#1e2130' }} }},
          y: {{ ticks: {{ color: '#9ca3af' }}, grid: {{ color: '#1e2130' }},
               title: {{ display: true, text: 'Diff (Actual − Proj)', color: '#6b7280' }} }}
        }}
      }}
    }});
    </script>
    """
    display(HTML(_chart))
else:
    print("No data for tag chart.")

## Ownership vs Diff Scatter

In [ ]:
if (not lineups.empty
    and "avg_own_pct" in lineups.columns
    and "diff" in lineups.columns):

    _tag_colors = {"SE Core": "#4a9eff", "SE Spicy": "#ef4444", "SE Alt": "#f59e0b"}
    _tag_col = "ricky_tag" if "ricky_tag" in lineups.columns else None

    _datasets = []

    # Untagged points
    if _tag_col:
        _unt = lineups[lineups[_tag_col].fillna("").str.strip().eq("")]
    else:
        _unt = lineups
    _unt_pts = [{"x": round(r["avg_own_pct"], 2), "y": round(r["diff"], 2)}
                for _, r in _unt.iterrows()]
    _datasets.append({
        "label": "Untagged",
        "data": _unt_pts,
        "backgroundColor": "#55555540",
        "borderColor": "#555555",
        "pointRadius": 3,
        "borderWidth": 0.5,
    })

    # Tagged points
    if _tag_col:
        for tag, color in _tag_colors.items():
            _sub = lineups[lineups[_tag_col] == tag]
            if _sub.empty:
                continue
            _pts = [{"x": round(r["avg_own_pct"], 2), "y": round(r["diff"], 2)}
                    for _, r in _sub.iterrows()]
            _datasets.append({
                "label": tag,
                "data": _pts,
                "backgroundColor": color + "cc",
                "borderColor": color,
                "pointRadius": 6,
                "borderWidth": 1.5,
            })

    import json as _json
    _ds_json = _json.dumps(_datasets, separators=(",", ":"))

    _chart = f"""
    <div style="background:#0f1117;border-radius:8px;padding:16px;max-width:800px">
      <div style="color:#e2e5ec;font-size:14px;font-weight:600;margin-bottom:10px">
        Lineup Ownership vs Performance</div>
      <canvas id="scatterChart" height="320"></canvas>
    </div>
    <script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.7/dist/chart.umd.min.js"></script>
    <script>
    new Chart(document.getElementById('scatterChart'), {{
      type: 'scatter',
      data: {{ datasets: {_ds_json} }},
      options: {{
        responsive: true,
        animation: {{ duration: 0 }},
        plugins: {{
          legend: {{ labels: {{ color: '#6b7280', font: {{ size: 11 }}, usePointStyle: true, boxWidth: 8 }} }}
        }},
        scales: {{
          x: {{ title: {{ display: true, text: 'Avg Ownership %', color: '#6b7280' }},
               ticks: {{ color: '#9ca3af' }}, grid: {{ color: '#1e2130' }} }},
          y: {{ title: {{ display: true, text: 'Diff (Actual − Proj)', color: '#6b7280' }},
               ticks: {{ color: '#9ca3af' }}, grid: {{ color: '#1e2130' }} }}
        }}
      }}
    }});
    </script>
    """
    display(HTML(_chart))
else:
    print("Missing columns for ownership scatter.")

---
### Notes
- Point `DATA_DIR` at any folder containing your Sim Lab CSV exports.
- Charts use Chart.js with the same dark palette as the Smash/Bust Explorer.
- All cells handle missing files gracefully — warnings print but nothing crashes.